In [ ]:
from parsers.lr.lr1 import LR1Parser
from parsers.lr.lr_parsing_engine import LREngine

from parsers.grammars import bison_to_ruleset

In [ ]:
bison = r"""
%token id assign then if else sem

%start P

%%

    P
        : L ;
    S 
        : id assign id
        | if id then S
        | if id then S else S
        ;
    L
        : S 
        | L sem S ;
"""


In [ ]:
p = LR1Parser(bison_to_ruleset(bison)).to_lalr()

In [ ]:
p.bison_like_report()

In [ ]:
len(p.states)

In [ ]:
print(p.to_tabulate())

In [ ]:
conflicts = p.get_conflicts()
state, sym, _ = conflicts[0]

cell = list(p.parsing_table[state][sym])
p.parsing_table[state][sym] = set(cell[1:])

In [ ]:
print(p.to_tabulate())

In [ ]:
e = LREngine(p)

In [ ]:
code_sample = """
if id then     
    if id then
        id assign id
    else
        id assign id sem
        id assign id $end
""".split()

In [ ]:
def print_iter(stack, states, symbol, action):
    for el in stack:
        print_el = next(iter(el.keys())) if isinstance(el, dict) else el
        print(print_el, end=", ")
    print(f" {symbol=}, {states[-1]=}, {action=}")

In [ ]:
import json

In [ ]:
out = e.parse(code_sample, iteration_callback=print_iter)

In [ ]:
from parsers.lr.helpers import pretify_stack

In [ ]:
print(pretify_stack(out))

In [ ]:
print(json.dumps(out, indent=2))